# 第 09 课 · 保存、恢复与可复现训练

恢复模型权重后可以继续推理；要让训练沿原来的轨迹继续，还需恢复优化器动量、随机数状态和训练步数。本课用一个小型 GPU 分类器比较连续训练与中断后恢复训练，直接检查最终参数是否一致。

checkpoint（检查点）是某个训练时刻保存的状态。为了集中观察保存与恢复机制，本课使用合成分类数据，不代表语言模型性能。只加载本课自己创建的可信 checkpoint；Python pickle 格式不应用于不可信来源。

## 1 · 连续运行、权重恢复、完整恢复三条路径

先训练 20 步保存，再连续训练到 40 步；然后从同一个保存点恢复，分别测试完整状态恢复和只恢复权重。保存是深拷贝，避免之后训练修改内存中的 checkpoint。

In [ ]:
import copy,json,random
from pathlib import Path
import numpy as np
import torch
from torch import nn
assert torch.cuda.is_available()
torch.set_num_threads(2)
torch.manual_seed(2030);torch.cuda.manual_seed_all(2030)
random.seed(2030);np.random.seed(2030)
data=torch.randn(256,8,device='cuda');labels=(data[:,0]+data[:,1]>0).long()
def new_model():return nn.Sequential(nn.Linear(8,32),nn.ReLU(),nn.Dropout(.2),nn.Linear(32,2)).cuda()
def advance(model,opt,count):
    model.train();losses=[]
    for _ in range(count):
        idx=torch.randint(len(data),(32,),device='cuda')
        loss=nn.functional.cross_entropy(model(data[idx]),labels[idx])
        opt.zero_grad(set_to_none=True);loss.backward();opt.step();losses.append(loss.item())
    return losses
model=new_model();opt=torch.optim.AdamW(model.parameters(),lr=.01)
advance(model,opt,20)
checkpoint=copy.deepcopy(dict(model=model.state_dict(),optimizer=opt.state_dict(),step=20,
    torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all(),
    python_rng=random.getstate(),numpy_rng=np.random.get_state()))
out=Path('support/experiment09_output');out.mkdir(exist_ok=True)
torch.save(checkpoint,out/'trusted_training_checkpoint.pt')
continuous_losses=advance(model,opt,20)
continuous=copy.deepcopy(model.state_dict())

def restore_rng(saved):
    torch.set_rng_state(saved['torch_rng']);torch.cuda.set_rng_state_all(saved['cuda_rng'])
    random.setstate(saved['python_rng']);np.random.set_state(saved['numpy_rng'])
saved=torch.load(out/'trusted_training_checkpoint.pt',weights_only=False,map_location=None)
restored=new_model();restored.load_state_dict(saved['model'])
restored_opt=torch.optim.AdamW(restored.parameters(),lr=.01);restored_opt.load_state_dict(saved['optimizer'])
restore_rng(saved)
restored_losses=advance(restored,restored_opt,20)
full_difference=max((continuous[k]-v).abs().max().item() for k,v in restored.state_dict().items())
assert full_difference<1e-6, f'恢复结果超出容差: {full_difference}'

weights_only=new_model();weights_only.load_state_dict(saved['model'])
fresh_opt=torch.optim.AdamW(weights_only.parameters(),lr=.01)
restore_rng(saved)
advance(weights_only,fresh_opt,20)
weights_difference=max((continuous[k]-v).abs().max().item() for k,v in weights_only.state_dict().items())
print('完整状态恢复后的最大参数差:',full_difference)
print('仅恢复权重后的最大参数差:',weights_difference)
assert weights_difference>full_difference
(out/'verification.json').write_text(json.dumps(dict(full_difference=full_difference,weights_only_difference=weights_difference,step=40),indent=2))

## 2 · 推理模式、显存与复现边界

`eval()` 关闭 dropout 等训练行为，不会自动关闭梯度；`inference_mode()` 关闭梯度记录。二者用途不同。删除引用后 `empty_cache()` 可归还未使用的缓存，但不能释放仍被变量引用的 GPU 张量。

本课在相同硬件、软件、数据与单进程设置下比较恢复。实际分布式训练还需保存学习率调度器、混合精度 scaler、数据采样器和每个 rank 的状态。不同 CUDA/库版本不保证逐位一致。

## 3 · 练习

把 dropout 从 0.2 改成 0.5，从头执行；再在独立副本中故意不恢复随机状态，观察差异。恢复训练前检查权重、优化器、随机状态和当前步数是否配套。

Notebook 中“重跑训练格”通常意味着继续训练；要复现同一起点，应重启内核并按顺序运行。

In [ ]:
restored.eval()
with torch.inference_mode():
    accuracy=(restored(data).argmax(-1)==labels).float().mean().item()
print(f'合成训练数据准确率: {accuracy:.1%}（不是独立测试）')
print(f'当前 PyTorch 已分配显存: {torch.cuda.memory_allocated()/2**20:.1f} MiB')
print('PASS：完整 checkpoint 恢复验证通过。')